<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Data Interoperability with Unity Catalog</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">5. Lakehouse Federation</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 5.2 Demo Federation to SQL Server and Snowflake

This demo walks through Lakehouse Federation from Unity Catalog to two external relational systems - **SQL Server** and **Snowflake** - using the same `CONNECTION` + `FOREIGN CATALOG` pattern for both. Each section is split into source-side steps (executed on SQL Server or Snowflake) and Databricks-side steps (executed in Unity Catalog).

## Learning Objectives

By the end of this demonstration, you will be able to:
- Provision a federation service principal / login on the source system with least-privilege access
- Create UC `CONNECTION` and `FOREIGN CATALOG` objects over SQL Server and Snowflake
- Run federated queries and explore foreign catalog metadata directly from Databricks
- Build cross-system joins that combine UC-resident data with foreign tables

<div style="font-size: 1em; border-left: 4px solid #7b1fa2; background: #f3e5f5; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #4a148c; font-size: 1.1em;">Video Demonstration</strong>
            <p style="margin: 8px 0 0 0; color: #333;">In the standard classroom environment, this demo is delivered as a <strong>video walkthrough</strong> because it requires external source systems and network connectivity not available in the lab environment. The notebook below contains the complete working demo. If you have the required infrastructure, you can run it end-to-end.</p>
        </div>
    </div>
</div>

<div style="font-size: 1em; border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Required Permissions</strong>
            <p style="margin: 8px 0 0 0; color: #333;">This demo configures federation connections on both platforms. The person running it needs:</p>
            <ul style="margin: 8px 0 0 16px; color: #333">
                <li><b>Databricks:</b> <b>Metastore Admin</b> or a principal with <code>CREATE CONNECTION</code> privilege.</li>
                <li><b>SQL Server:</b> A running SQL Server instance with <b>AdventureWorksDW</b> restored, and a login with read access to the target schemas.</li>
                <li><b>Snowflake:</b> <b><code>ACCOUNTADMIN</code></b> or a role with privileges to create databases and grant access.</li>
            </ul>
        </div>
    </div>
</div>

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #c62828; font-size: 1.1em;">Requires Network Connectivity</strong>
            <p style="margin: 8px 0 0 0; color: #333;">The Databricks workspace must have network connectivity to both the SQL Server instance and the Snowflake account. Source-side steps (marked with the SQL Server or Snowflake logo) run in the respective source console; Databricks-side steps (marked with the Databricks logo) run in this notebook.</p>
        </div>
    </div>
</div>

In [0]:
%run ../Includes/Classroom-Setup-Common

## A. Federation to SQL Server

The first half of the demo federates against a SQL Server instance hosting the **AdventureWorks** sample database. AdventureWorks is Microsoft's sample OLTP database; backups are available at <a href="https://github.com/Microsoft/sql-server-samples/releases/tag/adventureworks" target="_blank">github.com/Microsoft/sql-server-samples</a>.

Prerequisites:
- A SQL Server instance with the `AdventureWorksDW` database restored
- Network connectivity from your Databricks workspace to the SQL Server host
- Metastore admin or `CREATE CONNECTION` privilege in Unity Catalog

### A1. Create a Federation Login in SQL Server

Provision a dedicated SQL login for the federation connection so Unity Catalog can authenticate to SQL Server with least-privilege access. The pulldowns below show the T-SQL to run on the source system.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://devicon-website.vercel.app/api/microsoftsqlserver/plain.svg" width="20" height="20" style="vertical-align: middle;"> SQL Server:</span> Create Federation Login (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">Create a server-level login with a strong password:</p>
    <div class="code-block" data-language="sql">
-- Run on SQL Server (master database) as a sysadmin
CREATE LOGIN databricks_federation_login
    WITH PASSWORD = '<strong-password>',
         CHECK_POLICY = ON;
    </div>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://devicon-website.vercel.app/api/microsoftsqlserver/plain.svg" width="20" height="20" style="vertical-align: middle;"> SQL Server:</span> Create Database User and Grant Read (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">Map the login to a database user in <code>AdventureWorksDW</code> and grant read access:</p>
    <div class="code-block" data-language="sql">
-- Run on SQL Server (AdventureWorksDW database)
USE AdventureWorksDW;
CREATE USER databricks_federation_user
    FOR LOGIN databricks_federation_login;
ALTER ROLE db_datareader
    ADD MEMBER databricks_federation_user;
    </div>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://devicon-website.vercel.app/api/microsoftsqlserver/plain.svg" width="20" height="20" style="vertical-align: middle;"> SQL Server:</span> Verify Login Setup (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">Confirm the login and database user are configured correctly:</p>
    <div class="code-block" data-language="sql">
-- Verify the federation login and user
SELECT name, type_desc, is_disabled
FROM sys.server_principals
WHERE name = 'databricks_federation_login';
USE AdventureWorksDW;
SELECT dp.name AS user_name, r.name AS role_name
FROM sys.database_role_members rm
JOIN sys.database_principals dp ON rm.member_principal_id    = dp.principal_id
JOIN sys.database_principals r  ON rm.role_principal_id      = r.principal_id
WHERE dp.name = 'databricks_federation_user';
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var label = 'SQL Server';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

### A2. Create a Connection to SQL Server

Create a UC `CONNECTION` object that stores the host, port, and credentials for the SQL Server instance. The connection is created once and reused across foreign catalogs.

Prerequisites:
- **Host**: Your SQL Server hostname (e.g., `myserver.database.windows.net`)
- **Port**: Typically `1433`
- **User**: The SQL login created in Step A1 (e.g., `databricks_federation_login`)
- **Password**: The login password, ideally stored in a Databricks secret scope

<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <span style="font-size: 24px;">⚠️</span>
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Use Databricks Secrets in production</strong>
            <p style="margin: 8px 0 0 0; color: #333;">The example below uses a plaintext password placeholder for demo clarity. In production, replace with a secret reference: <code>password secret('scope', 'sqlserver_password')</code>.</p>
        </div>
    </div>
</div>

In [0]:
-- Create the connection object to your SQL Server instance
DROP CATALOG IF EXISTS sqlserver_adventureworks;
DROP CONNECTION IF EXISTS sqlserver_connection;

CREATE CONNECTION sqlserver_connection
TYPE sqlserver
OPTIONS (
    host '<your-sql-server-host>',         -- e.g. myserver.database.windows.net
    port '1433',
    user 'databricks_federation_login',
    password '<your-password>',            -- replace with secret('scope','key') in production 
    trustServerCertificate 'true'
);

In [0]:
-- Verify the connection object
DESCRIBE CONNECTION EXTENDED sqlserver_connection;

### A3. Create a Foreign Catalog over AdventureWorks

The foreign catalog mirrors the SQL Server database structure inside Unity Catalog. Once created, it appears alongside native UC catalogs and supports standard UC governance (grants, tags, lineage).

In [0]:
-- Create a foreign catalog that mirrors the AdventureWorksDW database
CREATE FOREIGN CATALOG IF NOT EXISTS sqlserver_adventureworks
USING CONNECTION sqlserver_connection
OPTIONS (database 'AdventureWorksDW');

In [0]:
DESCRIBE CATALOG EXTENDED sqlserver_adventureworks;

### A4. Explore the Federated Catalog

AdventureWorks uses named schemas (`dbo`, `HumanResources`, `Person`, `Production`, `Purchasing`, `Sales`) which map directly to UC schemas. Schema and table metadata is fetched from SQL Server on demand.

In [0]:
-- List schemas in the federated catalog
SHOW SCHEMAS IN sqlserver_adventureworks;

In [0]:
-- List tables in the dbo schema
SHOW TABLES IN sqlserver_adventureworks.dbo;

In [0]:
-- Inspect a federated table
DESCRIBE TABLE sqlserver_adventureworks.dbo.dimaccount;

### A5. Query Federated Data

Standard SQL works against foreign catalog tables. Filters, projections, and aggregates are pushed down to SQL Server where possible. The query below joins three SQL Server tables to produce internet sales by territory and calendar year.

In [0]:
-- Internet sales revenue by territory and calendar year
SELECT
  t.SalesTerritoryRegion              AS Territory,
  t.SalesTerritoryGroup               AS TerritoryGroup,
  d.CalendarYear,
  COUNT(*)                            AS OrderLines,
  ROUND(SUM(f.SalesAmount), 2)        AS TotalRevenue,
  ROUND(AVG(f.SalesAmount), 2)        AS AvgLineAmount
FROM sqlserver_adventureworks.dbo.factinternetsales f
JOIN sqlserver_adventureworks.dbo.dimsalesterritory t ON f.SalesTerritoryKey = t.SalesTerritoryKey
JOIN sqlserver_adventureworks.dbo.dimdate           d ON f.OrderDateKey      = d.DateKey
GROUP BY t.SalesTerritoryRegion, t.SalesTerritoryGroup, d.CalendarYear
ORDER BY d.CalendarYear, TotalRevenue DESC;

## B. Federation to Snowflake

The second half of the demo follows the same pattern against Snowflake. The Snowflake account in the lab has access to `SNOWFLAKE_SAMPLE_DATA`, which contains the `TPCH_SF1` schema we use to demonstrate cross-system joins between Snowflake and Databricks.

Prerequisites:
- A Snowflake account with a user that has appropriate privileges
- Network connectivity between Databricks and Snowflake (if Private Link or IP allowlisting are enabled)
- An RSA key pair for key-pair authentication (Snowflake is deprecating password-only auth for service accounts)

### B1. Generate an RSA Key Pair

Snowflake is deprecating password-only authentication for service accounts. Generate an RSA key pair locally - the public key goes on the Snowflake user, the private key is stored in a Databricks secret scope.

<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <span style="font-size: 24px;">⚠️</span>
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Password-Only Authentication Deprecated</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Snowflake is deprecating password-only authentication for service accounts. You must use <strong>key-pair authentication</strong> for programmatic access. This requires generating an RSA key pair and assigning the public key to your Snowflake user.</p>
        </div>
    </div>
</div>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    🔐 Generate RSA Key Pair (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">Run these commands in your terminal to generate the key pair:</p>
    <div class="code-block" data-language="bash">
# Generate private key (PKCS#8 format, no passphrase)
openssl genrsa 2048 | openssl pkcs8 -topk8 -nocrypt -inform PEM -out rsa_key.p8
<br/>
# Extract public key
openssl rsa -in rsa_key.p8 -pubout -out rsa_key.pub
<br/>
# Display public key (copy content between BEGIN and END lines for Snowflake)
cat rsa_key.pub | grep -v "BEGIN\|END" | tr -d '\n'
<br/>
# Display private key (copy content between BEGIN and END lines for the Databricks secret)
cat rsa_key.p8 | grep -v "BEGIN\|END" | tr -d '\n'
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var label = lang === 'bash' ? 'Terminal' : 'Snowflake';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

### B2. Create a Service User in Snowflake

Provision a dedicated Snowflake role and service user for the federation connection. Each pulldown below contains the SQL to run in Snowflake.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/snowflake/29B5E8" width="20" height="20" style="vertical-align: middle;"> Snowflake:</span> Create Federation Role (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">Create a dedicated role for Databricks federation access:</p>
    <div class="code-block" data-language="sql">
-- Create a role for the federation service account
CREATE ROLE IF NOT EXISTS DATABRICKS_FEDERATION_ROLE;
    </div>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/snowflake/29B5E8" width="20" height="20" style="vertical-align: middle;"> Snowflake:</span> Create Service User (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">Create a service user with no password (key-pair auth only):</p>
    <div class="code-block" data-language="sql">
-- Create the service user (TYPE = SERVICE disables password auth)
CREATE USER IF NOT EXISTS DATABRICKS_FEDERATION_USER
    TYPE = SERVICE
    DEFAULT_ROLE = DATABRICKS_FEDERATION_ROLE
    COMMENT = 'Service account for Databricks Lakehouse Federation';
    </div>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/snowflake/29B5E8" width="20" height="20" style="vertical-align: middle;"> Snowflake:</span> Grant Role to User (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">Assign the federation role to the service user:</p>
    <div class="code-block" data-language="sql">
-- Grant the role to the user
GRANT ROLE DATABRICKS_FEDERATION_ROLE TO USER DATABRICKS_FEDERATION_USER;
    </div>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/snowflake/29B5E8" width="20" height="20" style="vertical-align: middle;"> Snowflake:</span> Grant Database Privileges (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">Grant the necessary privileges to access the sample data (adjust names as needed):</p>
    <div class="code-block" data-language="sql">
-- Grant necessary privileges to the role
-- Using Snowflake's built-in sample data (SNOWFLAKE_SAMPLE_DATA)
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE DATABRICKS_FEDERATION_ROLE;
GRANT IMPORTED PRIVILEGES ON DATABASE SNOWFLAKE_SAMPLE_DATA TO ROLE DATABRICKS_FEDERATION_ROLE;
    </div>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/snowflake/29B5E8" width="20" height="20" style="vertical-align: middle;"> Snowflake:</span> Assign Public Key to User (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">Assign your RSA public key to the service user (paste the key content without BEGIN/END lines):</p>
    <div class="code-block" data-language="sql">
-- Assign the public key to the user
-- Paste your public key content (remove BEGIN/END lines and join into single line)
ALTER USER DATABRICKS_FEDERATION_USER SET RSA_PUBLIC_KEY = 'MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEA...your-key-here...';
    </div>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/snowflake/29B5E8" width="20" height="20" style="vertical-align: middle;"> Snowflake:</span> Verify User Setup (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">Verify the service user is configured correctly:</p>
    <div class="code-block" data-language="sql">
-- Verify the user setup
DESCRIBE USER DATABRICKS_FEDERATION_USER;
SHOW GRANTS TO USER DATABRICKS_FEDERATION_USER;
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var label = 'Snowflake';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

### B3. Create a Connection to Snowflake

Create a UC `CONNECTION` object that authenticates to Snowflake using key-pair authentication. The private key is referenced from a Databricks secret scope (`snowflake_migration / sf_private_key` in this example).

Prerequisites:
- **Host**: Your Snowflake account URL (e.g., `ORGNAME-ACCOUNTNAME.snowflakecomputing.com`)
- **User**: The Snowflake service user created in Step B2 (e.g., `DATABRICKS_FEDERATION_USER`)
- **Warehouse**: A Snowflake warehouse the service user has access to (e.g., `COMPUTE_WH`)
- **Role**: The role assigned to the service user (e.g., `DATABRICKS_FEDERATION_ROLE`)
- **Private Key**: The base64-encoded private key (contents of `rsa_key.p8` with headers removed and newlines stripped), stored as a Databricks secret

<div style="background:#e7f3fe;border-left:4px solid #2196F3;padding:16px;border-radius:4px;margin:16px 0;">
<strong>💡 Tip:</strong> Run this query in Snowflake to get your host URL:
<div class="code-block" data-language="sql">
-- Run in Snowflake to get your host URL
SELECT CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME() || '.snowflakecomputing.com';
</div>
</div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">Snowflake</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

In [0]:
-- Create the connection object to your Snowflake instance
DROP CATALOG IF EXISTS snowflake_sample_data;
DROP CONNECTION IF EXISTS snowflake_federation;

CREATE CONNECTION snowflake_federation
TYPE snowflake
OPTIONS (
    host '<your-account>.snowflakecomputing.com',  -- e.g. MGBHLAO-CY92030.snowflakecomputing.com
    port '443',
    user 'DATABRICKS_FEDERATION_USER',
    sfWarehouse 'COMPUTE_WH',
    sfRole 'DATABRICKS_FEDERATION_ROLE',
    pem_private_key secret('snowflake_migration','sf_private_key')
);

In [0]:
-- Verify the connection object
DESCRIBE CONNECTION EXTENDED snowflake_federation;

### B4. Create a Foreign Catalog over Snowflake's Sample Data

Point the foreign catalog at `SNOWFLAKE_SAMPLE_DATA`, which contains the `TPCH_SF1` schema we'll use for queries and the cross-system join.

In [0]:
DROP CATALOG IF EXISTS snowflake_sample_data;

-- Create a foreign catalog that mirrors the Snowflake sample database
CREATE FOREIGN CATALOG snowflake_sample_data
USING CONNECTION snowflake_federation
OPTIONS (database 'SNOWFLAKE_SAMPLE_DATA');

### B5. Explore the Federated Catalog

List Snowflake schemas and tables directly from Databricks. Metadata is fetched from Snowflake on demand.

In [0]:
-- List schemas in the federated catalog
SHOW SCHEMAS IN snowflake_sample_data;

In [0]:
-- List tables in the TPCH_SF1 schema
SHOW TABLES IN snowflake_sample_data.tpch_sf1;

In [0]:
-- Inspect a federated table
DESCRIBE TABLE snowflake_sample_data.tpch_sf1.nation;

### B6. Query Federated Data

Query Snowflake tables directly from Databricks without moving data. Filters and projections are pushed down to Snowflake.

In [0]:
-- Query a Snowflake table from Databricks
SELECT
    n_nationkey,
    n_name,
    n_regionkey,
    n_comment
FROM snowflake_sample_data.tpch_sf1.nation
LIMIT 25;

### B7. Cross-System Join (UC + Snowflake)

Join Snowflake-resident TPC-H `nation` with the equivalent Databricks `samples.tpch.nation` table to demonstrate a cross-platform join in a single SQL statement.

In [0]:
-- Join Snowflake TPCH data with Databricks samples TPCH data
SELECT
    sf.n_nationkey,
    sf.n_name AS snowflake_nation_name,
    uc.n_name AS databricks_nation_name,
    sf.n_regionkey
FROM snowflake_sample_data.tpch_sf1.nation sf
JOIN samples.tpch.nation uc
    ON sf.n_nationkey = uc.n_nationkey
ORDER BY sf.n_nationkey
LIMIT 25;

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <span style="font-size: 24px;">💡</span>
        <div>
            <strong style="color: #0d47a1; font-size: 1.1em;">Query Pushdown</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Lakehouse Federation optimises queries by pushing down predicates, projections, and aggregations to Snowflake where possible. This minimises data transfer and leverages Snowflake's compute for filtering and aggregations before results are returned to Databricks.</p>
        </div>
    </div>
</div>

## C. Inspect Pushdown

`EXPLAIN FORMATTED` reveals which filters and aggregations crossed the wire to each source. Aggregations on the Snowflake side become Snowflake SQL; filters on SQL Server become T-SQL; UC handles the join.

In [0]:
EXPLAIN FORMATTED
SELECT n_regionkey, COUNT(*)
FROM snowflake_sample_data.tpch_sf1.nation
WHERE n_nationkey > 5
GROUP BY n_regionkey;

In [0]:
EXPLAIN FORMATTED
SELECT t.SalesTerritoryRegion, COUNT(*)
FROM sqlserver_adventureworks.dbo.factinternetsales f
JOIN sqlserver_adventureworks.dbo.dimsalesterritory t ON f.SalesTerritoryKey = t.SalesTerritoryKey
WHERE f.OrderDateKey BETWEEN 20120101 AND 20121231
GROUP BY t.SalesTerritoryRegion;

<div style="font-size: 1em; border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #0d47a1; font-size: 1.1em;">Pushdown differences</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Both connectors do predicate pushdown - that is the floor they share. What is different is how much <em>else</em> they push:</p>
            <ul style="margin: 8px 0 0 16px; color: #333">
                <li><strong>Snowflake</strong> hands the <em>whole plan</em> over: filter, projection, <code>GROUP BY</code>, and <code>COUNT</code> all run on Snowflake. Spark receives the final aggregated rows. The plan collapses to one <code>SnowflakePlan</code> node and the rewritten SQL appears as <code>External engine query:</code>.</li>
                <li><strong>SQL Server</strong> pushes <em>filter and projection only</em>. The join and the aggregation run locally in Photon. The plan is a full 22-node Photon graph; <code>PushedDownOperators(...)</code> on each <code>Scan JDBC</code> row shows exactly which predicates crossed the wire.</li>
            </ul>
            <p style="margin: 8px 0 0 0; color: #333;">The practical consequence is data volume on the wire. Snowflake returns a handful of aggregated rows. SQL Server returns full filtered fact + dimension rows and Photon does the join and aggregation locally - so the selectivity of the filters that <em>did</em> push is what governs how much data arrives.</p>
        </div>
    </div>
</div>

## Key Takeaways

Lakehouse Federation lets Unity Catalog reach into external relational systems with a single uniform pattern - the same `CONNECTION` and `FOREIGN CATALOG` mechanics work for SQL Server, Snowflake, and every other supported source.

<div style="font-size: 1em; border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #2e7d32; font-size: 1.1em;">What This Demo Showed</strong>
            <ul style="margin: 8px 0 0 16px; color: #333">
                <li><strong><code>CONNECTION</code> + <code>FOREIGN CATALOG</code></strong> is the same pattern regardless of source - SQL Server, Snowflake, or any other source UC supports</li>
                <li>Source-side provisioning (logins, roles, key-pair auth) is decoupled from the Databricks-side configuration; both halves are needed but each is owned by the team that owns the source</li>
                <li>UC governs federation: connections are securable objects, foreign catalogs inherit access control, and audit covers federated queries</li>
                <li><strong>Cross-platform joins</strong> across UC, SQL Server, and Snowflake live in one SQL statement</li>
                <li>The query planner pushes filters and aggregations down to each source - <code>EXPLAIN FORMATTED</code> confirms what crossed the wire</li>
            </ul>
        </div>
    </div>
</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>